# OpenSearch Neural Search (OCI Data Science)

This notebook demonstrates how to configure an OpenSearch cluster for neural (vector) search using the OpenSearch ML plugins and k-NN support. It covers:

- Connecting to an existing OpenSearch cluster (host, port, user, password supplied via environment variables)
- Updating ML-related cluster settings
- Registering and deploying an ML model for text embeddings
- Creating ingest pipelines that compute embeddings at index time
- Creating a k-NN index mapping to store vector embeddings
- Indexing documents and performing keyword, neural, and hybrid searches

Prerequisites
- An OpenSearch cluster reachable from this environment with ML plugins enabled
- Environment variables set: OPENSEARCH_HOST, OPENSEARCH_PORT, OPENSEARCH_USER, OPENSEARCH_PASS
- The `opensearch-py` Python package installed in the kernel

Security
- Do not commit credentials to source control. Use environment variables or secret stores.

Quick start
1. Set the environment variables (example shown in a cell below).
2. Install dependencies (`pip install opensearch-py`).
3. Run the cells in order.

Notes
- This notebook hasn't been executed in this environment during the last save; run cells to make sure the cluster is reachable and API calls succeed.

---


### OCI Data Science - Useful Tips
<details>
<summary><font size="2">Check for Public Internet Access</font></summary>

```python
import requests
response = requests.get("https://oracle.com")
assert response.status_code==200, "Internet connection failed"
```
</details>
<details>
<summary><font size="2">Helpful Documentation </font></summary>
<ul><li><a href="https://docs.cloud.oracle.com/en-us/iaas/data-science/using/data-science.htm">Data Science Service Documentation</a></li>
<li><a href="https://docs.cloud.oracle.com/iaas/tools/ads-sdk/latest/index.html">ADS documentation</a></li>
</ul>
</details>
<details>
<summary><font size="2">Typical Cell Imports and Settings for ADS</font></summary>

```python
%load_ext autoreload
%autoreload 2
%matplotlib inline

import warnings
warnings.filterwarnings('ignore')

import logging
logging.basicConfig(format='%(levelname)s:%(message)s', level=logging.ERROR)

import ads
from ads.dataset.factory import DatasetFactory
from ads.automl.provider import OracleAutoMLProvider
from ads.automl.driver import AutoML
from ads.evaluations.evaluator import ADSEvaluator
from ads.common.data import ADSData
from ads.explanations.explainer import ADSExplainer
from ads.explanations.mlx_global_explainer import MLXGlobalExplainer
from ads.explanations.mlx_local_explainer import MLXLocalExplainer
from ads.catalog.model import ModelCatalog
from ads.common.model_artifact import ModelArtifact
```
</details>
<details>
<summary><font size="2">Useful Environment Variables</font></summary>

```python
import os
print(os.environ["NB_SESSION_COMPARTMENT_OCID"])
print(os.environ["PROJECT_OCID"])
print(os.environ["USER_OCID"])
print(os.environ["TENANCY_OCID"])
print(os.environ["NB_REGION"])
```
</details>

In [ ]:
# Set environment variables (example). In practice, set these securely outside the notebook.
%env OPENSEARCH_PORT=9200
%env OPENSEARCH_HOST=zcxxxx.com
%env OPENSEARCH_USER=zzxxx
%env OPENSEARCH_PASS=zzxxx

# Quick runtime check: ensure required env vars exist
import os
for var in ["OPENSEARCH_HOST", "OPENSEARCH_PORT", "OPENSEARCH_USER", "OPENSEARCH_PASS"]:
    if var not in os.environ:
        raise EnvironmentError(f"Required environment variable {var} is not set")


In [ ]:
# Install the OpenSearch client library. Run this cell if the package is not present in the environment.
# Note: In some managed notebook environments, you may need to restart the kernel after installing packages.
!pip install opensearch-py --quiet


In [ ]:
from opensearchpy import OpenSearch, RequestsHttpConnection
from requests.auth import HTTPBasicAuth
import os
import json
import time
from typing import Dict, Any

# OpenSearch Configuration
INDEX_NAME = "neural_search"
MODEL_NAME = "huggingface/sentence-transformers/msmarco-distilbert-base-tas-b"
MODEL_VERSION = "1.0.1"
MODEL_DIM = 768
SEARCH_PIPELINE_NAME = "neural_search_pipeline"
INGEST_PIPELINE_NAME = "neural_search_ingest_pipeline"
DATA_FILE = "data.json"

# Connect to OpenSearch using environment variables
client = OpenSearch(
    hosts=[{"host": os.environ["OPENSEARCH_HOST"], "port": int(os.environ.get("OPENSEARCH_PORT", 9200))}],
    http_auth=HTTPBasicAuth(os.environ["OPENSEARCH_USER"], os.environ["OPENSEARCH_PASS"]),
    use_ssl=True,
    verify_certs=True,
    connection_class=RequestsHttpConnection,
)


def update_cluster_settings() -> None:
    """Update ML-related cluster settings used by OpenSearch ML plugins.

    This is a non-destructive call but may change how ML tasks are scheduled and executed.
    """
    cluster_settings = {
        "persistent": {
            "plugins.ml_commons.only_run_on_ml_node": "false",
            "plugins.ml_commons.model_access_control_enabled": "true",
            "plugins.ml_commons.native_memory_threshold": "99"
        }
    }

    # Update Cluster Settings
    response = client.cluster.put_settings(body=cluster_settings)
    print("ML related cluster settings updated successfully:", response)


def register_model_group() -> str:
    """Register a model group used to organize related ML models."""
    model_group_payload = {
        "name": "Neural_model_group3",
        "description": "A model group for NLP models",
        "access_mode": "public"
    }

    response = send_request("/_plugins/_ml/model_groups/_register", "POST", model_group_payload)
    model_group_id = response.get("model_group_id", "N/A")
    print(f"Model Group Registered Successfully. ID: {model_group_id}")
    return model_group_id


def register_model_to_model_group(model_groupid: str) -> str:
    """Register a specific model inside an existing model group.

    Returns a long-running task id that should be polled until completion.
    """
    model_payload = {
        "name": MODEL_NAME,
        "version": MODEL_VERSION,
        "model_group_id": model_groupid,
        "model_format": "TORCH_SCRIPT"
    }

    response = send_request("/_plugins/_ml/models/_register", "POST", model_payload)
    task_id = response.get("task_id", "N/A")
    print(f"Model Registered Successfully. Task ID: {task_id}")
    return task_id


def is_task_completed(task_id: str) -> str:
    """Check the status of a long-running ML task.

    If the task is completed, returns the created model id. Otherwise returns 'N/A'.
    """
    response = send_request(f"/_plugins/_ml/tasks/{task_id}", "GET")
    state = response.get("state", "N/A")
    print("Task status:", state)
    if state == "COMPLETED":
        return response.get("model_id", "N/A")
    return "N/A"


def deploy_model(model_id: str) -> str:
    """Deploy a registered model to create an inference endpoint on the cluster.

    Returns a task id that should be polled to confirm deployment.
    """
    response = send_request(f"/_plugins/_ml/models/{model_id}/_deploy", "POST")
    return response.get("task_id", "N/A")


def create_ingest_pipeline(model_id: str, pipeline_name: str) -> None:
    """Create an ingest pipeline that adds embeddings to documents at index time."""
    model_group_payload = {
      "description": "An NLP ingest pipeline",
      "processors": [
        {
          "text_embedding": {
            "model_id": model_id,
            "field_map": {
              "content": "passage_embedding"
            }
          }
        }
      ]
    }
    send_request(f"/_ingest/pipeline/{pipeline_name}", "PUT", model_group_payload)


def create_search_pipeline(pipeline_name: str) -> None:
    """Create a post-processing search pipeline to normalize and combine results for hybrid search."""
    payload = {
      "description": "Post processor for hybrid search",
      "phase_results_processors": [
        {
          "normalization-processor": {
            "normalization": {
              "technique": "min_max"
            },
            "combination": {
              "technique": "arithmetic_mean",
              "parameters": {
                "weights": [
                  0.3,
                  0.7
                ]
              }
            }
          }
        }
      ]
    }
    response = send_request(f"/_search/pipeline/{pipeline_name}", "PUT", payload)
    print("Search pipeline created:", response)


def create_kNN_index(pipeline_name: str) -> None:
    """Create a k-NN enabled index with mapping for passage embeddings."""
    index_mapping = {
      "settings": {
        "index.knn": "true",
        "default_pipeline": pipeline_name
      },
      "mappings": {
        "properties": {
          "passage_embedding": {
            "type": "knn_vector",
            "dimension": MODEL_DIM,
            "method": {
              "engine": "lucene",
              "space_type": "l2",
              "name": "hnsw",
              "parameters": {}
            }
          },
          "title": {"type": "text"},
          "description": {"type": "text"},
          "category": {"type": "text"},
          "content": {"type": "text"}
        }
      }
    }

    if client.indices.exists(INDEX_NAME):
        print(f"Index '{INDEX_NAME}' already exists. Deleting and recreating...")
        client.indices.delete(index=INDEX_NAME)

    response = client.indices.create(index=INDEX_NAME, body=index_mapping)
    print(f"Index '{INDEX_NAME}' created successfully:", response)


def ingest_documents_from_file(data_file: str) -> None:
    """Read documents from a JSON file and index them into OpenSearch."""
    with open(data_file, "r", encoding="utf-8") as file:
        documents = json.load(file)

        for doc in documents:
            document_body = {
                "title": doc.get("title"),
                "description": doc.get("description"),
                "content": doc.get("content"),
                "category": doc.get("category"),
            }

            response = client.index(index=INDEX_NAME, body=document_body)
            print(f"Document indexed with ID: {response['_id']}")


def ingest_documents() -> None:
    """Index a small in-memory dataset (useful for demos)."""
    data = [
    {
        "title": "AI in Healthcare",
        "description": "Artificial Intelligence is transforming healthcare by automating diagnoses, enhancing patient care, and optimizing hospital operations.",
        "category": "Healthcare",
        "content": "AI in healthcare uses machine learning models to predict health trends, assist in medical imaging, and improve decision-making processes."
    },
    {
        "title": "The Future of Autonomous Vehicles",
        "description": "Self-driving cars are revolutionizing the transportation industry with AI-driven decision-making, reducing accidents, and improving traffic flow.",
        "category": "Technology",
        "content": "Autonomous vehicles are equipped with sensors, cameras, and AI algorithms to navigate roads without human intervention, making transportation safer and more efficient."
    },
    {
        "title": "Blockchain Technology Explained",
        "description": "Blockchain is a decentralized ledger technology that enables secure and transparent transactions without the need for intermediaries.",
        "category": "Finance",
        "content": "Blockchain allows digital transactions to be recorded securely across a network of computers, ensuring data integrity and preventing fraud in financial services."
    },
    {
        "title": "Quantum Computing for Beginners",
        "description": "Quantum computing promises to revolutionize fields like cryptography and optimization by using quantum bits (qubits) for complex calculations.",
        "category": "Science",
        "content": "Quantum computers use the principles of quantum mechanics to perform calculations that would be impossible for classical computers, making them useful for scientific research and security."
    },
    {
        "title": "Sustainable Energy Solutions",
        "description": "Sustainable energy sources, such as solar and wind, are helping reduce our dependence on fossil fuels and mitigate climate change.",
        "category": "Environment",
        "content": "Renewable energy sources like solar panels and wind turbines generate power in an environmentally friendly way, contributing to global sustainability and energy security."
    }
    ]

    for document in data:
        response = client.index(
            index=INDEX_NAME,
            body=document
        )
    print(f"Document indexed with ID: {response['_id']}")


def display_results(response: Dict[str, Any]) -> None:
    """Pretty print search results returned from OpenSearch."""
    print("Search Results:")
    for hit in response['hits']['hits']:
        print(f"Title: {hit['_source'].get('title')}")
        print(f"Description: {hit['_source'].get('description')}")
        print(f"Content: {hit['_source'].get('content')}\n")


def search(query: Dict[str, Any]) -> None:
    """Execute a raw search query against the configured index."""
    response = client.search(
        index=INDEX_NAME,
        body=query
    )
    display_results(response)


def keyword_search(text: str) -> None:
    """Perform a regular text match search."""
    query = {
      "_source": {
        "excludes": [
          "passage_embedding"
        ]
      },
      "query": {
        "match": {
          "content": {
            "query": text
          }
        }
      }
    }
    search(query)


def neural_search(text: str, model_id: str) -> None:
    """Perform a neural (vector) search using a deployed model."""
    query = {
      "_source": {
        "excludes": [
          "passage_embedding"
        ]
      },
      "query": {
        "neural": {
          "passage_embedding": {
            "query_text": text,
            "model_id": model_id,
            "k": 1
          }
        }
      }
    }
    search(query)


def hybrid_search(text: str, txt_query: str, model_id: str) -> None:
    """Combine text match and neural search using a search pipeline for better relevance."""
    search_query = {
      "_source": {
        "excludes": [
          "passage_embedding"
        ]
      },
      "query": {
        "hybrid": {
          "queries": [
            {
              "match": {
                "content": {
                  "query": txt_query
                }
              }
            },
            {
              "neural": {
                "passage_embedding": {
                  "query_text": text,
                  "model_id": model_id,
                  "k": 1
                }
              }
            }
          ]
        }
      }
    }
    response = send_request(f"/{INDEX_NAME}/_search?search_pipeline={SEARCH_PIPELINE_NAME}", "GET", search_query)
    display_results(response)


def send_request(p_url: str, p_method: str, p_body: Any = {}) -> Any:
    """Helper wrapper around the low-level OpenSearch transport to perform HTTP-style calls.

    Use this for ML plugin endpoints that are not covered by the high-level client.
    """
    response = client.transport.perform_request(
        method=p_method,
        url=p_url,
        body=p_body
    )
    return response


def delete_item(i_path: str, verb: str = "DELETE") -> None:
    """Delete resources using the low-level transport API."""
    response = client.transport.perform_request(
        method=verb,
        url=i_path
    )


def clean_up() -> None:
    """Remove created resources (index, pipelines, models, model groups).

    Note: Variables `model_id` and `group_id` must exist in this scope for undeploy calls to succeed.
    """
    if client.indices.exists(INDEX_NAME):
        print(f"Index '{INDEX_NAME}' already exists. Deleting ...")
        client.indices.delete(index=INDEX_NAME)

    delete_item(f"/_search/pipeline/{SEARCH_PIPELINE_NAME}")
    delete_item(f"/_ingest/pipeline/{INGEST_PIPELINE_NAME}")
    # The following calls assume `model_id` and `group_id` are defined after running registration/deployment steps.
    try:
        delete_item(f"/_plugins/_ml/models/{model_id}/_undeploy", "POST")
        delete_item(f"/_plugins/_ml/models/{model_id}")
        delete_item(f"/_plugins/_ml/model_groups/{group_id}")
    except NameError:
        print("model_id or group_id not found in the current scope; skipping model cleanup")


In [ ]:
# Kick off the registration and update flow. These calls make changes on the OpenSearch cluster.
print("Updating cluster settings")
update_cluster_settings()

print("register_model_group")
group_id = register_model_group()

print("register_model_to_model_group")
task_id = register_model_to_model_group(group_id)

# Note: `task_id` is a long running task id. Use `is_task_completed` to poll for completion.


Updating cluster settings
ML related cluster settings updated successfully: {'acknowledged': True, 'persistent': {'plugins': {'ml_commons': {'only_run_on_ml_node': 'false', 'model_access_control_enabled': 'true', 'native_memory_threshold': '99'}}}, 'transient': {}}
register_model_group
Model Group Registered Successfully. ID: {'model_group_id': 'FyYZ-JQBIEbv8ErnRWqA', 'status': 'CREATED'}
register_model_to_model_group
Model Registered Successfully. ID: {'task_id': 'GCYZ-JQBIEbv8ErnRWqS', 'status': 'CREATED'}


In [ ]:
# Poll the registration task status. This will print the task state; wait until it reports COMPLETED.
is_task_completed(task_id)
print("Continue only when this task state is COMPLETED.")


{'model_id': 'vvcZ-JQBVCrLV2EoRvPI', 'task_type': 'REGISTER_MODEL', 'function_name': 'TEXT_EMBEDDING', 'state': 'COMPLETED', 'worker_node': ['EbF3KRuaRtWELB6DxnKF5w'], 'create_time': 1739329193362, 'last_update_time': 1739329225884, 'is_async': True}
Continue only when this task state is COMPLETED.


In [ ]:
# After the registration task completes, poll for the created model id and then deploy the model.
# Replace Sleep(...) which is undefined with time.sleep(seconds)

print("is_task_completed " + str(task_id))
model_id = is_task_completed(task_id)
if model_id == "N/A":
    print("Task not completed yet. Sleeping for 10s before retrying")
    time.sleep(10)

print("Model id: ", model_id)

# Deploy the model and poll for deployment completion
d_task_id = deploy_model(model_id)
if d_task_id == "N/A":
    print("Deployment task not returned; sleeping for 10s")
    time.sleep(10)

print("create_ingest_pipeline")
create_ingest_pipeline(model_id, INGEST_PIPELINE_NAME)

print("create_search_pipeline")
create_search_pipeline(SEARCH_PIPELINE_NAME)


is_task_completed GCYZ-JQBIEbv8ErnRWqS
{'model_id': 'vvcZ-JQBVCrLV2EoRvPI', 'task_type': 'REGISTER_MODEL', 'function_name': 'TEXT_EMBEDDING', 'state': 'COMPLETED', 'worker_node': ['EbF3KRuaRtWELB6DxnKF5w'], 'create_time': 1739329193362, 'last_update_time': 1739329225884, 'is_async': True}
Model id vvcZ-JQBVCrLV2EoRvPI
create_ingest_pipeline
create_search_pipeline


In [ ]:
# Create the k-NN index and ingest documents from a file.
print("create_kNN_index")
create_kNN_index(INGEST_PIPELINE_NAME)

print("ingest_documents from file")
ingest_documents_from_file(DATA_FILE)


create_kNN_index
Index 'neural_search' already exists. Deleting and recreating...
Index 'neural_search' created successfully: {'acknowledged': True, 'shards_acknowledged': True, 'index': 'neural_search'}
ingest_documents from file
Document indexed with ID: xPcm-JQBVCrLV2EoAPNr
Document indexed with ID: xfcm-JQBVCrLV2EoAPPm
Document indexed with ID: xvcm-JQBVCrLV2EoAfNi
Document indexed with ID: x_cm-JQBVCrLV2EoAfPl
Document indexed with ID: yPcm-JQBVCrLV2EoAvNh


In [ ]:
# Example keyword search
text = "How does AI help in diagnosing diseases?"
keyword_search(text)


Search Results:
Title: AI in Healthcare
Description: Artificial Intelligence is transforming healthcare by automating diagnoses, enhancing patient care, and optimizing hospital operations.
Content: AI in healthcare uses machine learning models to predict health trends, assist in medical imaging, and improve decision-making processes.

Title: The Future of Autonomous Vehicles
Description: Self-driving cars are revolutionizing the transportation industry with AI-driven decision-making, reducing accidents, and improving traffic flow.
Content: Autonomous vehicles are equipped with sensors, cameras, and AI algorithms to navigate roads without human intervention, making transportation safer and more efficient.

Title: Blockchain Technology Explained
Description: Blockchain is a decentralized ledger technology that enables secure and transparent transactions without the need for intermediaries.
Content: Blockchain allows digital transactions to be recorded securely across a network of compute

In [ ]:
# Another keyword search example
text = "predict health trends"
keyword_search(text)


Search Results:
Title: AI in Healthcare
Description: Artificial Intelligence is transforming healthcare by automating diagnoses, enhancing patient care, and optimizing hospital operations.
Content: AI in healthcare uses machine learning models to predict health trends, assist in medical imaging, and improve decision-making processes.



In [ ]:
# Neural search using the deployed model. Ensure `model_id` is available in the session (from previous deployment steps).
text = "How does AI help in diagnosing diseases?"
neural_search(text, model_id)


Search Results:
Title: AI in Healthcare
Description: Artificial Intelligence is transforming healthcare by automating diagnoses, enhancing patient care, and optimizing hospital operations.
Content: AI in healthcare uses machine learning models to predict health trends, assist in medical imaging, and improve decision-making processes.



In [ ]:
# Hybrid search example: combines keyword and neural results using the configured search pipeline.
text = "How does AI help in diagnosing diseases?"
q_text = "predict health trends"
hybrid_search(text, q_text, model_id)


Search Results:
Title: AI in Healthcare
Description: Artificial Intelligence is transforming healthcare by automating diagnoses, enhancing patient care, and optimizing hospital operations.
Content: AI in healthcare uses machine learning models to predict health trends, assist in medical imaging, and improve decision-making processes.

